<a href="https://colab.research.google.com/github/SergiSama/UIC-CRB1-2026-2027/blob/main/m4_image_basics_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 4 · Notebook 1 — Image Processing Basics (OpenCV)
### Computing, Robotics & Bionics · companion to A. Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch*

A **self-contained** computer-vision reference (Géron ships no vision-tools notebook). It treats the
biomedical image as a NumPy array and builds the classical pipeline: colour spaces, point operations and
histograms, geometric warps (the Module 2 maps in homogeneous coordinates), spatial filtering and
**convolution** (the core of a CNN — Géron Ch. 12), and edge detection.

Run in **Google Colab** (*Runtime → Run all*). OpenCV and the sample image ship pre-installed (offline).

**Contents**
1. Images as arrays (and the BGR/RGB trap)
2. Colour spaces: RGB, grayscale, HSV
3. Point operations and histograms
4. Geometric transformations
5. Spatial filtering and convolution
6. Edges: Sobel and Canny

Most sections end with an **Exercise**; run the **Solution** cell to check.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
print("OpenCV", cv2.__version__)

---
## 1 · Images as arrays (and the BGR/RGB trap)

An image is just an array: grayscale is `(H, W)`, colour is `(H, W, 3)`, dtype `uint8` (0–255), indexed
`img[row, col] = img[y, x]` with the origin at the **top-left**. **Pitfall:** `cv2.imread` returns
**BGR**, while Matplotlib and scikit-learn use **RGB** — convert with `cv2.cvtColor` or channels look
swapped. (Here the sample image arrives as RGB from scikit-learn.)

In [ ]:
from sklearn.datasets import load_sample_image
rgb = load_sample_image("flower.jpg")          # RGB uint8 (offline)
print("shape:", rgb.shape, "| dtype:", rgb.dtype, "| min/max:", rgb.min(), rgb.max())

# slicing crops; this is a pure array operation
crop = rgb[50:250, 150:400]
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(rgb); ax[0].set_title(f"full image {rgb.shape}"); ax[0].axis("off")
ax[1].imshow(crop); ax[1].set_title(f"crop {crop.shape}"); ax[1].axis("off")
plt.show()

In [ ]:
# The BGR/RGB trap, made visible: showing BGR data as if RGB swaps red<->blue
bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(rgb);  ax[0].set_title("correct (RGB)"); ax[0].axis("off")
ax[1].imshow(bgr);  ax[1].set_title("BGR shown as RGB -> wrong colours"); ax[1].axis("off")
plt.show()

---
## 2 · Colour spaces: RGB, grayscale, HSV

Collapsing to **grayscale** keeps shape/texture (most pipelines work here). **HSV** separates hue,
saturation and value, which makes segmenting by colour robust to lighting — we preview a value (intensity)
mask, the idea behind the cell segmentation in Notebook 2.

In [ ]:
gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)   # sklearn gives RGB -> use RGB2GRAY
hsv  = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
print("gray shape:", gray.shape, "| hsv shape:", hsv.shape)

fig, ax = plt.subplots(1, 4, figsize=(14, 3.2))
ax[0].imshow(gray, cmap="gray"); ax[0].set_title("grayscale")
for i, name in enumerate(["Hue", "Saturation", "Value"]):
    ax[i+1].imshow(hsv[:, :, i], cmap="gray"); ax[i+1].set_title(name)
for a in ax: a.axis("off")
plt.show()

---
## 3 · Point operations and histograms

A **point operation** remaps each pixel independently. Linear contrast/brightness is the affine map
`I' = alpha*I + beta`. The **histogram** shows the intensity distribution; **equalisation** spreads it to
boost contrast.

In [ ]:
alpha, beta = 1.6, 10                              # contrast x1.6, +10 brightness
adjusted = cv2.convertScaleAbs(gray, alpha=alpha, beta=beta)   # clips safely at 255
equalised = cv2.equalizeHist(gray)

fig, ax = plt.subplots(2, 3, figsize=(12, 6))
for a, img, t in zip(ax[0], [gray, adjusted, equalised], ["original", "contrast/brightness", "equalised"]):
    a.imshow(img, cmap="gray"); a.set_title(t); a.axis("off")
for a, img in zip(ax[1], [gray, adjusted, equalised]):
    a.hist(img.ravel(), bins=64, color="steelblue"); a.set_xlim(0, 255)
ax[1,0].set_ylabel("pixel count")
plt.tight_layout(); plt.show()

**Exercise 3.** Build a high-contrast version with `alpha=2.5, beta=-120` and show its histogram.
What happens to the darkest and brightest pixels (clipping)?

> 🤖 *Gemini tip:* "Explain how cv2.convertScaleAbs applies alpha and beta to pixel values, and why extreme settings clip pixels to 0 and 255."

In [ ]:
# Your code here

---
## 4 · Geometric transformations

Geometric transforms move pixel **positions** — the linear maps of Module 2 written in homogeneous
coordinates as a 2x3 affine matrix, applied with `cv2.warpAffine`. This is the basis of image
**registration** (aligning scans).

In [ ]:
h, w = gray.shape
# rotation about the centre by 30 deg, with scaling 0.9
M_rot = cv2.getRotationMatrix2D(center=(w/2, h/2), angle=30, scale=0.9)
rotated = cv2.warpAffine(gray, M_rot, (w, h))
# translation by (60, 40): a 2x3 affine matrix
M_tr = np.float32([[1, 0, 60], [0, 1, 40]])
translated = cv2.warpAffine(gray, M_tr, (w, h))
print("rotation matrix (2x3):\n", M_rot.round(3))

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
for a, img, t in zip(ax, [gray, rotated, translated], ["original", "rotate 30, scale 0.9", "translate (60,40)"]):
    a.imshow(img, cmap="gray"); a.set_title(t); a.axis("off")
plt.show()

---
## 5 · Spatial filtering and convolution

**Convolution** slides a kernel over the image and outputs the weighted sum beneath it. Positive kernels
that sum to 1 **blur** (denoise); a centre-positive/surround-negative kernel **sharpens**. A CNN uses the
same operation but *learns* the kernels (Géron Ch. 12).

In [ ]:
# add noise, then denoise with blur kernels
noisy = np.clip(gray.astype(np.int16) + np.random.default_rng(0).normal(0, 25, gray.shape), 0, 255).astype(np.uint8)
box = cv2.blur(noisy, (5, 5))                       # 5x5 averaging
gauss = cv2.GaussianBlur(noisy, (5, 5), sigmaX=1.5) # weighted, edge-preserving-ish
sharpen_kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32)
sharp = cv2.filter2D(gray, -1, sharpen_kernel)      # arbitrary kernel via filter2D

fig, ax = plt.subplots(1, 4, figsize=(14, 3.2))
for a, img, t in zip(ax, [noisy, box, gauss, sharp], ["noisy", "box blur 5x5", "Gaussian blur", "sharpen"]):
    a.imshow(img, cmap="gray"); a.set_title(t); a.axis("off")
plt.show()

**Exercise 5.** A 3x3 averaging (box) kernel has every entry equal to `1/9`. Build it with NumPy, apply
it with `cv2.filter2D`, and confirm the result matches `cv2.blur(gray, (3,3))`.

> 🤖 *Gemini tip:* "Explain 2-D convolution with cv2.filter2D, and why a 3x3 kernel of ones divided by nine is the same as a box blur."

In [ ]:
# Your code here

---
## 6 · Edges: Sobel and Canny

An **edge** is a large image gradient. **Sobel** kernels estimate the x- and y-derivatives (Module 3); the
**magnitude** highlights boundaries. **Canny** refines this into clean thin contours — one call.

In [ ]:
sx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)   # d/dx
sy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)   # d/dy
mag = np.sqrt(sx**2 + sy**2)
mag = (255 * mag / mag.max()).astype(np.uint8)
edges = cv2.Canny(gray, 80, 160)

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
for a, img, t in zip(ax, [np.abs(sx), mag, edges], ["|Sobel x| (vertical edges)", "gradient magnitude", "Canny edges"]):
    a.imshow(img, cmap="gray"); a.set_title(t); a.axis("off")
plt.show()

---
### You now have the vision toolkit
Images as arrays, colour spaces, point operations and histograms, geometric warps, convolution/filtering,
and edges. **Next:** Notebook 2 uses thresholding, morphology and contours to **segment and count cells**
on real Broad BBBC microscopy images.